# Wolf Sheep Predation — Comparing Two BehaviorSpace Runs

Comparing **100 sheep / 100 wolves** vs **400 sheep / 400 wolves** — same model code (unmodified), same parameter grid (`sheep-reproduce`, `wolf-reproduce`, `grass-regrowth-time`), same time limit (1500 ticks).

**Before running:** upload the two BehaviorSpace table CSVs to this Colab session (left sidebar → Files → Upload), and update the filenames in the cell below if they don't match.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

## 1. Load the data

BehaviorSpace table exports start with 6 lines of metadata (model name, timestamp, world dimensions) before the actual header row, so we skip them with `skiprows=6`. We also rename the columns to something easier to work with in code — the raw column names come out as things like `[run number]` and `sheep-reproduce`, which are awkward to reference in pandas.

In [ ]:
def read_bs_table(path):
    df = pd.read_csv(path, skiprows=6)
    df.columns = ["run_number", "sheep_reproduce", "wolf_reproduce",
                  "grass_regrowth_time", "step",
                  "land_with_grass", "count_sheep", "count_wolves"]
    return df

f_100 = "experiment_100_100-table.csv"
f_400 = "experiment_400_400-table.csv"

exp_100 = read_bs_table(f_100)
exp_400 = read_bs_table(f_400)

exp_100.info()

## 2. Derived variables

None of these are in the raw BehaviorSpace output — we compute them from `count_sheep`, `count_wolves`, and `step`:

- **`censored`**: the run hit the time limit (1500 ticks) without resolving. We don't actually know what would have happened afterward — only that the ecosystem survived at least that long.
- **`wolves_survived`**: at least one wolf was still alive when the run ended.
- **`total_collapse`**: both sheep and wolves were gone by the end (the ecosystem fully collapsed).

In [ ]:
def add_derived(df, time_limit=1500):
    df = df.copy()
    df["censored"] = df["step"] == time_limit
    df["sheep_survived"] = df["count_sheep"] > 0
    df["wolves_survived"] = df["count_wolves"] > 0
    df["total_collapse"] = (~df["sheep_survived"]) & (~df["wolves_survived"])
    return df

exp_100 = add_derived(exp_100)
exp_400 = add_derived(exp_400)

exp_100["config"] = "100 sheep / 100 wolves"
exp_400["config"] = "400 sheep / 400 wolves"

both = pd.concat([exp_100, exp_400], ignore_index=True)
both.head()

## 3. Descriptive summary

For each configuration: what fraction of runs ended in total collapse, what fraction still had at least one wolf, and what fraction were censored (hit the time limit).

In [ ]:
summary_table = both.groupby("config")[["total_collapse", "wolves_survived", "censored"]].mean()
summary_table

**How to read this table:** if `wolves_survived` is 0.0 (or very close to it) in *both* configurations, that tells you wolf extinction isn't something that goes away just because you scale the whole population up — which is exactly the question this comparison is designed to probe. Don't just glance at the numbers — compare them to what we already found when testing other population sizes.

## 4. How long do ecosystems survive, in each configuration?

In [ ]:
both.groupby("config")["step"].describe()

**How to read this table:** compare the medians (50%) and the spread (25%–75%). A configuration where most runs end quickly, with a tight spread, behaves very differently from one where survival time varies a lot depending on the parameters. Also check the `max` — if it's close to or exactly 1500, that configuration has runs that were censored (see `censored` above).

## 5. Heatmap — probability of total collapse, by parameters

Each cell shows the proportion of runs (averaged over the 3 repetitions) that ended in total collapse, for a given combination of `sheep-reproduce` and `wolf-reproduce`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, (config_name, sub) in zip(axes, both.groupby("config")):
    pivot = sub.pivot_table(index="wolf_reproduce", columns="sheep_reproduce",
                             values="total_collapse", aggfunc="mean")
    sns.heatmap(pivot, ax=ax, cmap="RdYlGn_r", vmin=0, vmax=1,
                annot=True, fmt=".2f", cbar=(ax is axes[-1]))
    ax.set_title(config_name)
    ax.set_xlabel("sheep-reproduce (%)")
    ax.set_ylabel("wolf-reproduce (%)")

plt.suptitle("Probability of total ecosystem collapse")
plt.tight_layout()
plt.show()

**How to read this heatmap:** does the `100/100` panel show a gradient — some parameter combinations clearly safer (green) than others (red) — while `400/400` looks mostly uniform and red no matter what the parameters are? That would mean that at high population density, the parameters barely matter anymore: the system collapses almost regardless of what you set `sheep-reproduce` or `wolf-reproduce` to. That's a genuinely different qualitative regime, not just "the same story, faster."

## 6. Survival time distribution, by configuration

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=both, x="config", y="step")
sns.stripplot(data=both, x="config", y="step", color="black", alpha=0.15, size=3)
plt.ylabel("Ticks until collapse or time limit (1500)")
plt.xlabel("")
plt.title("Ecosystem survival time, by configuration")
plt.show()

**How to read this plot:** if the `400/400` box is short and sits low on the y-axis (most runs collapse fast, little variation between them), while `100/100` is tall and spread out (survival time depends a lot on the parameters), that confirms the pattern from the heatmap — just from a different angle on the same data. Two independent views agreeing with each other is a good sign the pattern is real, not an artifact of how you chose to summarize it.

## 7. Wolf survival rate, side by side

In [ ]:
wolf_surv = both.groupby("config")["wolves_survived"].mean()

plt.figure(figsize=(6, 4))
wolf_surv.plot(kind="bar", color="steelblue")
plt.ylabel("Proportion of runs with at least 1 wolf alive at the end")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.title("Wolf survival, by configuration")
plt.show()

## 8. Discussion questions for class

- Did scaling both populations up (100→400 sheep, 100→400 wolves, same 1:1 ratio) change wolf survival at all? Compare this bar chart to the other population sizes we already tested.
- Does the `400/400` heatmap still show meaningful differences between parameter combinations, or does population density alone dominate the outcome, regardless of `sheep-reproduce` / `wolf-reproduce`?
- Looking only at the `100/100` heatmap: which corner (high/low sheep-reproduce, high/low wolf-reproduce) is safest for the ecosystem? Does that match the direction we found earlier — that higher `sheep-reproduce` makes collapse *more* likely, not less?
- Why might a denser population collapse in a way that's less sensitive to the reproduction-rate parameters than a sparser one? (Hint: think about how much grass and how many patches are available per animal in each case — the world size doesn't change, only the population does.)